# This notebook outputs layer similarities to help decide which layers to trim
Code source for Part 3: https://github.com/arcee-ai/PruneMe/tree/main

## 1. Set up

In [ ]:
#load utils
from google.colab import files
uploaded = files.upload()  # This will open a file picker

Saving utils.py to utils.py


In [ ]:
!pip install -q transformers
!pip install -q torchinfo
!pip install -q datasets
!pip install -q evaluate
!pip install -q transformers[torch]
!pip install -q peft
!pip install -q accelerate
!pip install bitsandbytes
#!pip install -U bitsandbytes


import logging
import csv
import argparse
import numpy as np
from tqdm import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModel, AutoModelForSequenceClassification
import datasets
from datasets import load_dataset

from utils import get_last_non_padded_tokens, compute_block_distances
from typing import Optional

logging.basicConfig(level=logging.INFO)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-c

In [ ]:
import logging
import csv
import argparse
import numpy as np
from tqdm import tqdm


import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModel, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import datasets
from datasets import load_dataset

from utils import get_last_non_padded_tokens, compute_block_distances
from typing import Optional

logging.basicConfig(level=logging.INFO)

## 2. Load Data

In [ ]:
# Load the dataset
moral_dataset = load_dataset("demelin/moral_stories", "cls-action+context+consequence-norm_distance")

# Select only the first 4000 for training and first 1000 for testing
moral_train_dataset = moral_dataset['train'].shuffle().select(range(20000))
moral_dev_dataset = moral_dataset['test'].shuffle().select(range(2000))

README.md:   0%|          | 0.00/12.0k [00:00<?, ?B/s]

moral_stories.py:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/374k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/374k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def preprocess_moral(data):
    merged_texts = []

    for moral, immoral in zip(data['moral_action'], data['immoral_action']):
        # Ignore "not specified" values for moral and immoral actions
        if moral.lower() == "not specified":
            action_text = immoral
        elif immoral.lower() == "not specified":
            action_text = moral
        else:
            action_text = f"{moral} {immoral}"  # Merge both sentences

        # Add the [CLS] token at the beginning, followed by situation, norm, and action_text, all separated by [SEP]
        #action_text = f"[CLS] {action_text} [SEP]"

        merged_texts.append(action_text)  # Store the merged text

    return merged_texts

In [ ]:
type(preprocess_moral(moral_train_dataset))

list

In [ ]:
MAX_SEQUENCE_LENGTH = 128

## 3. Main function for layer similarity calculation

In [ ]:
def main(model_path: str, dataset, batch_size: int, max_length: int,
         layers_to_skip: int, dataset_size: Optional[int] = None):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # if resource is a problem
    quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                            bnb_4bit_use_double_quant=True,
                                            bnb_4bit_quant_type="nf4",
                                            bnb_4bit_compute_dtype=torch.bfloat16)

    model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                 device_map="auto",
                                                 quantization_config=quantization_config,
                                                 output_hidden_states=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()

    #dataset = datasets.load_dataset(dataset, split=dataset_subset)
    #if dataset_size:
    #    dataset = dataset.select(range(dataset_size))

    dataloader = DataLoader(preprocess_moral(dataset), batch_size=batch_size, shuffle=False, drop_last=True) # torch object

    # Initialize a list to store distances for each block across the dataset
    all_distances = [[] for _ in range(model.config.num_hidden_layers - layers_to_skip)]


    for batch in tqdm(dataloader, desc="Processing batches"):
        inputs = tokenizer(batch, return_tensors="pt", padding="longest", max_length=max_length, truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        attention_mask = inputs["attention_mask"]
        hidden_states = outputs.hidden_states
        last_non_padded_hidden_states = get_last_non_padded_tokens(hidden_states, attention_mask)

        # Remove the first element to account for the input layer not being considered a model hidden layer
        # This adjustment is necessary for analyses focusing on the model's internal transformations
        last_non_padded_hidden_states = last_non_padded_hidden_states[1:]

        # Ensure that the length of last_non_padded_hidden_states matches the number of model hidden layers minus one
        assert len(last_non_padded_hidden_states) == model.config.num_hidden_layers, "Length of last_non_padded_hidden_states  \
        does not match expected number of hidden layers."

        # Compute distances and append to all_distances
        distances = compute_block_distances(last_non_padded_hidden_states, layers_to_skip)
        for i, distance in enumerate(distances):
            all_distances[i].append(distance)

    # Calculate average distances for each block
    average_distances = [np.mean(block_distances) for block_distances in all_distances]

    # Write the average distances to a CSV file and compute the minimum average distance
    min_distance = float('inf')  # Initialize with infinity
    min_distance_layer = 0  # Initialize with an impossible value
    data = []

    for i, avg_dist in enumerate(average_distances):
        data.append({
            'block_start': i + 1,  # layer indices are 1-based in the paper
            'block_end': i + 1 + layers_to_skip,
            'average_distance': avg_dist
        })
        if avg_dist < min_distance:
            min_distance = avg_dist
            min_distance_layer = i + 1

    return pd.DataFrame(data)


In [ ]:
df = main(model_path='roberta-base', dataset=moral_dev_dataset, batch_size=8, max_length=128, layers_to_skip=2, dataset_size=2000)

for i in range(4, 10,2):
  print("layer to skip " + str(i))
  new_df = main(model_path='roberta-base', dataset=moral_dev_dataset, batch_size=8, max_length=128, layers_to_skip=i, dataset_size=2000)
  df = pd.concat([df, new_df])

df

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

layer to skip 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing batches: 100%|██████████| 250/250 [00:10<00:00, 24.44it/s]


layer to skip 8


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing batches: 100%|██████████| 250/250 [00:10<00:00, 24.14it/s]


,block_start,block_end,average_distance
0,1,3,0.125971
1,2,4,0.093126
2,3,5,0.089655
3,4,6,0.086866
4,5,7,0.088413
5,6,8,0.118889
6,7,9,0.105235
7,8,10,0.099256
8,9,11,0.104377
9,10,12,0.102506


## 4. Layer Prunning and Heal

In [ ]:
!pip install -q evaluate
import evaluate

In [ ]:
torch.manual_seed(2947)
np.random.seed(2947)

In [ ]:
# Load accuracy and F1 metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)  # Convert logits to class predictions

    # Compute accuracy
    acc = metric_acc.compute(predictions=predictions, references=labels)

    # Compute F1-score (weighted to account for class imbalance)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")

    # Return both metrics
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def trim_middle_layers_roberta(layer_start=8, layer_end=14):
    # Load model + tokenizer
    model = AutoModelForSequenceClassification.from_pretrained("roberta-base")

    total_layers = len(model.roberta.encoder.layer)

    # Validation
    if layer_start < 0 or layer_end > total_layers or layer_start > layer_end:
        raise ValueError(f"Invalid range: start={layer_start}, end={layer_end}, model has {total_layers} layers.")

    # Keep layers before and after the middle chunk
    new_layers = (
        model.roberta.encoder.layer[:layer_start] +  # Early layers
        model.roberta.encoder.layer[layer_end + 1:]  # Late layers
    )

    model.roberta.encoder.layer = new_layers

    # Update model config
    model.config.num_hidden_layers = len(new_layers)

    print(f"Trimmed out layers {layer_start} to {layer_end}. Remaining layers: {len(new_layers)}")
    return model

In [ ]:
def tokenize_function(examples, tokenizer):
    return tokenizer(preprocess_moral(examples),
                     padding="max_length", truncation=True, max_length=MAX_SEQUENCE_LENGTH)

In [ ]:
# Tokenize
train_dataset = moral_train_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
test_dataset = moral_dev_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

# Set correct columns
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])



Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# number of layer trimmed = 2

for i in range(2, 14-2, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+2)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 4. Remaining layers: 9


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

...

Trimmed out layers 8 to 10. Remaining layers: 9


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.416200,0.468501,0.790500,0.789876


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 10 to 12. Remaining layers: 10


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.421300,0.456731,0.799500,0.798555


In [ ]:
# number of layer trimmed = 4

for i in range(2, 14-4, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+4)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 6. Remaining layers: 7


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.490500,0.522505,0.761000,0.760654


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 4 to 8. Remaining layers: 7


...

Trimmed out layers 6 to 10. Remaining layers: 7


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.424600,0.470809,0.793500,0.793033


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 8 to 12. Remaining layers: 8


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.425600,0.473494,0.789000,0.788077


In [ ]:
# number of layer trimmed = 6

for i in range(2, 14-6, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+6)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 8. Remaining layers: 5


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.503900,0.533346,0.746000,0.745854


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 4 to 10. Remaining layers: 5


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.469900,0.503304,0.773000,0.772519


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 6 to 12. Remaining layers: 6


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.451400,0.493974,0.771000,0.770595


## 5. Getting inference time

In [ ]:
def preprocess_moral(data, tokenizer):
    merged_texts = []

    for moral, immoral in zip(data['moral_action'], data['immoral_action']):
        # Ignore "not specified" values for moral and immoral actions
        if moral.lower() == "not specified":
            action_text = immoral
        elif immoral.lower() == "not specified":
            action_text = moral
        else:
            action_text = f"{moral} {immoral}"  # Merge both sentences

        # Add the [CLS] token at the beginning, followed by situation, norm, and action_text, all separated by [SEP]
        #action_text = f"[CLS] {action_text} [SEP]"

        merged_texts.append(action_text)  # Store the merged text

    # Tokenize the batch of merged sentences
    encoded = tokenizer.batch_encode_plus(
        merged_texts,
        max_length=MAX_SEQUENCE_LENGTH,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "token_type_ids": encoded["token_type_ids"]
    }


In [ ]:
import torch
import time
from torch.utils.data import Subset, DataLoader
import random

# Prepare the training and validation data
def preprocess_data(dataset, tokenizer):
    # Preprocess the data and tokenize it
    return dataset.map(preprocess_moral, batched=True, fn_kwargs={'tokenizer': tokenizer})
# Custom collate function to convert batch items to tensors
def collate_fn(batch):
    # Stack the batch and ensure it's in tensor format
    input_ids = torch.stack([torch.tensor(item['input_ids']) for item in batch])
    attention_mask = torch.stack([torch.tensor(item['attention_mask']) for item in batch])
    token_type_ids = torch.stack([torch.tensor(item['token_type_ids']) for item in batch])
    labels = torch.tensor([item['labels'] for item in batch])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'token_type_ids': token_type_ids, 'labels': labels}
# Format dataset to match model input expectations
def format_dataset(example):
    return {
        "input_ids": example["input_ids"],
        "attention_mask": example["attention_mask"],
        "token_type_ids": example["token_type_ids"],
        "labels": example["label"] if "label" in example else 0  # Default to 0 if missing
    }
# Function to measure inference time on N samples
def measure_inference_time(model, dataloader, device, sample_size=100):
    model.eval()
    # Sample random examples from the dataset
    indices = random.sample(range(len(dataloader.dataset)), sample_size)
    sampled_dataset = Subset(dataloader.dataset, indices)
    sampled_dataloader = DataLoader(sampled_dataset, batch_size=8, collate_fn=collate_fn)
    # Warm-up (optional but good practice on GPU)
    for _ in range(5):
        for batch in sampled_dataloader:
            with torch.no_grad():
                model(batch['input_ids'].to(device),
                      attention_mask=batch['attention_mask'].to(device))
    # Accurate timing
    torch.cuda.synchronize()
    start_time = time.time()
    for batch in sampled_dataloader:
        with torch.no_grad():
            model(batch['input_ids'].to(device),
                  attention_mask=batch['attention_mask'].to(device))
    torch.cuda.synchronize()
    total_time = time.time() - start_time
    avg_time_per_batch = total_time / len(sampled_dataloader)
    print(f"\nModel: {model.__class__.__name__}")
    print(f"Total inference time on {sample_size} samples: {total_time:.4f} seconds")
    print(f"Average inference time per batch: {avg_time_per_batch:.4f} seconds")
    return total_time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
formatted_dev_data = preprocess_data(moral_dev_dataset, tokenizer).map(format_dataset, remove_columns=moral_dev_dataset.column_names)
dev_dataloader = DataLoader(formatted_dev_data, batch_size=8, collate_fn=collate_fn)


# Example usage — test inference time for student model only
inference_time = measure_inference_time(model, dev_dataloader, device, sample_size=500)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Model: RobertaForSequenceClassification
Total inference time on 500 samples: 1.4549 seconds
Average inference time per batch: 0.0231 seconds
Total Parameters: 124,647,170


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for i in range(2, 8, 2):
  print("./roberta_base_"+str(12-i)+"_layers")
  model = trim_middle_layers_roberta(layer_start=12-i, layer_end=12)
  model.to(device)
  inference_time = measure_inference_time(model, dev_dataloader, device, sample_size=500)
  total_params = sum(p.numel() for p in model.parameters())
  print(f"Total Parameters: {total_params:,}")

./roberta_base_10_layers


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trimmed out layers 10 to 12. Remaining layers: 10


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model: RobertaForSequenceClassification
Total inference time on 500 samples: 1.2386 seconds
Average inference time per batch: 0.0197 seconds
Total Parameters: 110,471,426
./roberta_base_8_layers
Trimmed out layers 8 to 12. Remaining layers: 8


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model: RobertaForSequenceClassification
Total inference time on 500 samples: 0.9913 seconds
Average inference time per batch: 0.0157 seconds
Total Parameters: 96,295,682
./roberta_base_6_layers
Trimmed out layers 6 to 12. Remaining layers: 6

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 0.7526 seconds
Average inference time per batch: 0.0119 seconds
Total Parameters: 82,119,938
